# Module 04 — Checkpointing

**Released:** v1.14.0 → v1.14.3 (April 2026)

Checkpointing persists the **entire live runtime** at event boundaries. With it you can:

- **Resume** a run that was killed mid-flight (OOM, Ctrl-C, transient LLM error).
- **Fork** a run — restart from step N with different inputs; new writes track back to the source via `parent_id`.
- **Audit** what the system looked like at any step: inputs, LLM calls so far, tool outputs, pending state.

It works on three kickoffs: `Flow.kickoff(from_checkpoint=…)`, `Crew.kickoff(from_checkpoint=…)`, and `Agent.kickoff(from_checkpoint=…)` (async in notebooks).

This notebook walks through:

1. How it actually works (event listener → serialize → provider)
2. The `CheckpointConfig` knobs that matter
3. The event vocabulary (incl. signals like `SIGINT`)
4. Providers: `JsonProvider` vs `SqliteProvider`
5. **Step-by-step: Crew kickoff with checkpointing + resume** (`checkpoint_flow.py`)
6. **Step-by-step: `Agent.kickoff()` with checkpointing + resume**
7. Forking with lineage (Crew — pick a snapshot *before* the last task finishes)
8. Extending: custom providers
9. CLI reference

## 1. How it works under the hood

```
┌────────────────┐    event fires    ┌──────────────────────┐
│  Flow / Crew / │──────────────────▶│  checkpoint_listener │
│  Agent runtime │                   └──────────┬───────────┘
└────────────────┘                              │
                                                ▼
             ┌──────────────────────────────────────────────────────┐
             │  for every active entity with a CheckpointConfig:    │
             │   if event type ∈ config.on_events (or "*"):         │
             │     data = RuntimeState.model_dump_json()            │
             │     id   = provider.checkpoint(                      │
             │              data, config.location,                  │
             │              parent_id=<prev id>, branch="main")     │
             │     if config.max_checkpoints: provider.prune(...)   │
             └──────────────────────────────────────────────────────┘
```

Key details:

- **`RuntimeState` is global.** Serializing it captures *every* active crew / flow / agent — not just the one that fired the event. That's why resuming a single flow can rehydrate its agents, pending tasks, and LLM context together.
- **Handlers are lazy.** The listener only hooks into the event bus the first time a `CheckpointConfig` is resolved. Zero overhead when nothing is checkpointed.
- **Writes are cheap.** `SqliteProvider` uses WAL; `JsonProvider` writes one file per checkpoint. Pruning is per-branch and happens inline after each write.
- **Lineage is on the row.** Every checkpoint stores `parent_id` and `branch`, so the full tree (main → fork → fork-of-fork) is queryable later via `crewai checkpoint info`.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Same config the Module 04 crew demo uses (repo-root ``.checkpoints/``, JsonProvider).
from showcase.flows.checkpoint_flow import CHECKPOINT_ROOT, checkpoint_config

config = checkpoint_config()
config

## 2. `CheckpointConfig` — the five fields

| Field | Default | Purpose |
|---|---|---|
| `location` | `./.checkpoints` | Directory (JsonProvider) or DB file path (SqliteProvider). Created on first write. |
| `provider` | required | `JsonProvider()` or `SqliteProvider()` — or any `BaseProvider` subclass. |
| `on_events` | `["task_completed"]` | Event types that trigger a write. `["*"]` captures everything. |
| `max_checkpoints` | `None` | Keep at most N per branch; older rows are pruned after each write. |
| `restore_from` | `None` | Checkpoint id, file path, or `"db#id"` location. Setting this turns `kickoff(from_checkpoint=config)` into a **resume** instead of a fresh run. |

## 3. The event vocabulary

`on_events` accepts any `CheckpointEventType` literal. There are ~100 of them — here are the groupings that matter:

| Group | Examples | When you'd pick these |
|---|---|---|
| **Crew lifecycle** | `crew_kickoff_started/completed/failed`, `task_started/completed/failed` | Coarse-grained checkpointing between task boundaries. |
| **Flow lifecycle** | `flow_started/finished/paused`, `method_execution_started/finished/failed/paused` | One checkpoint per Flow step. |
| **Agent lifecycle** | `agent_execution_started/completed/error`, `lite_agent_execution_*` | Track every Agent invocation; `lite_*` events fire for `Agent.kickoff()`. |
| **LLM** | `llm_call_started/completed/failed`, `llm_stream_chunk`, `llm_guardrail_*` | Debug LLM behaviour, retry from last successful call. |
| **Tools** | `tool_usage_started/finished`, `tool_execution_error` | Checkpoint before/after every tool call — great for long-running tool chains. |
| **Plan-and-Execute** | `step_observation_*`, `plan_refinement`, `plan_replan_triggered`, `goal_achieved_early` | Granular snapshots of the planner's internal loop (Module 02). |
| **Memory** | `memory_save_*`, `memory_query_*`, `memory_retrieval_*` | Audit trail of what was read / written to memory. |
| **Signals** | `SIGINT`, `SIGTERM`, `SIGHUP`, `SIGTSTP`, `SIGCONT` | ⭐ **Checkpoint on Ctrl-C** — the listener intercepts the signal, snapshots, then re-raises. |
| **Env sentinels** | `cc_env`, `codex_env`, `cursor_env`, `default_env` | Fire when an agent session boots inside a particular coding-agent context. |

Common picks:

- `["task_completed"]` — safest default.
- `["task_completed", "task_failed"]` — post-mortem on failures.
- `["method_execution_finished"]` — Flow-step granularity.
- `["task_completed", "SIGINT", "SIGTERM"]` — normal completion + graceful-exit safety net.
- `["*"]` — everything. Big, but unbeatable for debugging.

## 4. Providers — `JsonProvider` vs `SqliteProvider`

| | `JsonProvider` (default) | `SqliteProvider` |
|---|---|---|
| Layout | One `.json` file per checkpoint in `location/` | Single `.db` file with WAL journaling |
| Best for | Dev, manual inspection (`cat checkpoint_xxx.json`) | Many checkpoints per run; bulk queries via SQL |
| Locking | Filesystem-level | `PRAGMA journal_mode=WAL` — concurrent readers OK |
| Lineage | `parent_id` field inside each JSON | `parent_id` column alongside `branch` |

Both share the same `BaseProvider` contract (6 methods: `checkpoint`, `acheckpoint`, `from_checkpoint`, `afrom_checkpoint`, `extract_id`, `prune`), so swapping providers is a one-line change.

## 5. Step-by-step — checkpoint a **Crew**

[`checkpoint_flow.py`](../src/showcase/flows/checkpoint_flow.py) builds a **three-task** crew with `{topic}` / `{audience}` inputs:

1. **Research** — bullet facts only  
2. **Tune for `{audience}`** — audience-fit draft (context: research)  
3. **Improve tone** — line edit only (context: tuned draft)

Checkpoints use **JsonProvider** at `CHECKPOINT_ROOT` (repo `.checkpoints/`, branch subdirs like `main/`) with `on_events=["task_completed"]` — **one JSON file per finished task** (three per full run).

`run_fresh()` attaches `checkpoint=checkpoint_config()` on the `Crew` and calls `kickoff(inputs=…)` — that is a *fresh run with checkpointing on*, not a resume.

**Prereq:** `ANTHROPIC_API_KEY` (or `OPENAI_API_KEY` with `SHOWCASE_LLM=openai`), `PYTHONPATH=src` (or install the package) so `showcase.*` imports work.

### Step 1 — run it fresh

In [ ]:
from showcase.flows.checkpoint_flow import run_fresh

crew = run_fresh()
print(crew.tasks[-1].output.raw)

### Step 2 — list what got written

After a full run you should see **three** new JSON rows on `main` (one per task). If you Ctrl-C mid-crew, you will see fewer files under `.checkpoints/main/`. Newest files are best found with `list_crew_checkpoint_files()` or the CLI.

In [ ]:
from showcase.flows.checkpoint_flow import list_crew_checkpoint_files

for p in list_crew_checkpoint_files(limit=8):
    print(p)

# CLI (point at the same directory the crew uses):
# !crewai checkpoint --location ./.checkpoints list 2>&1 | tail -20

### Step 3 — resume from a specific checkpoint

Copy a **full path** to one of the JSON files from Step 2 (or use `list_crew_checkpoint_files()[0]` while iterating). `resume_from` uses `Crew.from_checkpoint` under the hood.

On the resumed run, **tasks that already completed keep their outputs**; only pending tasks call the LLM again.

```python
from showcase.flows.checkpoint_flow import (
    forkable_crew_checkpoint_paths,
    list_crew_checkpoint_files,
    resume_from,
)

# Prefer a snapshot that still has pending work (e.g. after Ctrl-C mid-crew).
pending = forkable_crew_checkpoint_paths(limit=10)
if pending:
    ck = str(pending[0])
else:
    files = list_crew_checkpoint_files(limit=1)
    assert files, "Run Step 1 first."
    ck = str(files[0])  # often a finished run — resume may do little LLM work

resumed = resume_from(ck)
print(resumed.tasks[-1].output.raw)
```

Use `list_crew_checkpoint_files()` if you already know which JSON path you want.

**SQLite:** if you use `SqliteProvider`, pass `restore_from` as `"./.checkpoints.db#<id>"` — the provider splits on `#`.

**Try this:** run `run_fresh()`, interrupt after the first or second task, then `resume_from("<path-to-that-json>")` — completed tasks stay skipped.

→ See §7 for **forking** (new branch + optional `inputs=` overrides). **Forking from the very last checkpoint** (all tasks done) does nothing useful — `fork_from` raises unless you pass `allow_completed=True`.

## 6. Step-by-step — checkpoint an **`Agent.kickoff()`**

Single agents can checkpoint too. This is useful when an agent is a long-running loop (tool calls, multi-turn reasoning) and you want restartability without wrapping it in a Crew or Flow.

### Step 1 — build an agent and kick it off with checkpointing

The same `CheckpointConfig` shape works — `Agent.kickoff()` accepts `from_checkpoint=...` just like `Crew.kickoff()`.

In [ ]:
from crewai import Agent
from crewai.state.checkpoint_config import CheckpointConfig
from crewai.state.provider.sqlite_provider import SqliteProvider
from showcase.shared import get_llm

# Separate SQLite file for this cell so it does not mix with the Crew JSON tree.
CHECKPOINT_DB = "./.checkpoints_agent.db"

agent = Agent(
    role="Architecture Analyst",
    goal="Explain a piece of architecture clearly",
    backstory="Senior engineer who writes crisp technical explainers.",
    llm=get_llm(),
)

fresh = CheckpointConfig(
    location=CHECKPOINT_DB,
    provider=SqliteProvider(),
    # lite_agent_* are the events Agent.kickoff fires
    on_events=["lite_agent_execution_completed"],
    max_checkpoints=5,
)

# Inside Jupyter `agent.kickoff()` returns a coroutine — top-level await unwraps it.
first = await agent.kickoff(
    "In 80 words, explain why WAL-mode SQLite is a good fit for a per-process "
    "checkpoint store. Focus on concurrency and crash safety.",
    from_checkpoint=fresh,
)
print(first.raw)

### Step 2 — resume with a new input (context threading)

Pass a **different** `messages=` argument to a second `agent.kickoff()` with `restore_from` set. One subtlety worth flagging up front: the snapshot captures the agent's **config** (role, goal, tools, usage counters, memory binding, checkpoint config) — **not** the prior kickoff's conversation scratchpad. To give the resumed call that context, thread the prior output into the new prompt yourself.

**What the resume still buys you:**
- Identical agent config + tool state (usage counters carry over, so rate-limited tools don't reset).
- Lineage tracking via `parent_id` — every new checkpoint on the resumed call points at the snapshot you resumed from.
- Same DB, same pruning rules, same branch label.

**What you handle manually:**
- Prior conversation context — thread `first.raw` (or whatever subset you need) into the new prompt.

For multi-turn agents that need persistent memory across many kickoffs, give the agent a `Memory` (see [Module 03](./03_unified_memory.ipynb)); the memory recall handles cross-turn context so you don't have to thread outputs by hand.

In [ ]:
import sqlite3

# Grab the latest agent checkpoint id — in a real session you'd usually pick
# one from `crewai checkpoint list` output.
with sqlite3.connect(CHECKPOINT_DB) as db:
    ck_id = db.execute(
        "SELECT id FROM checkpoints "
        "WHERE json(data) LIKE '%Architecture Analyst%' "
        "ORDER BY rowid DESC LIMIT 1"
    ).fetchone()[0]
print("resuming from:", ck_id)

resume = CheckpointConfig(
    location=CHECKPOINT_DB,
    provider=SqliteProvider(),
    on_events=["lite_agent_execution_completed"],
    max_checkpoints=5,
    restore_from=f"{CHECKPOINT_DB}#{ck_id}",
)

# Thread the first output into the follow-up so the agent has grounded context.
second = await agent.kickoff(
    f"You previously wrote:\n\n{first.raw}\n\n"
    "Now name TWO specific failure modes to watch for in that design. "
    "One sentence each.",
    from_checkpoint=resume,
)
print("\n---\n")
print(second.raw)

In [ ]:
# Agent checkpoints live in CHECKPOINT_DB (separate from the Crew JSON demo).
!crewai checkpoint --location ./.checkpoints_agent.db list 2>&1 | tail -10

## 7. Forking with lineage (Crew)

Forking = restore from a checkpoint on a **new branch**, optionally merge **new kickoff `inputs`** (e.g. a different `{audience}`). The provider records `parent_id` and `branch` on each new write so trees stay navigable (`crewai checkpoint` TUI / `info`).

[`fork_from`](../src/showcase/flows/checkpoint_flow.py) wraps `Crew.fork(cfg, branch=...)` then `kickoff(inputs=...)`.

**Why the “final” checkpoint is a dead end:** if every task already has output, there is nothing left for the crew to execute — fork or resume both no-op on LLM work. Fork **after research** (1/3 tasks) or **after audience tune** (2/3) when you want a new branch with e.g. `inputs={"audience": "…"}`. If you pass a terminal snapshot by mistake, `fork_from` raises `ValueError` (override with `allow_completed=True` only if you know what you are doing).

```python
from showcase.flows.checkpoint_flow import fork_from, forkable_crew_checkpoint_paths

paths = forkable_crew_checkpoint_paths(limit=10)
assert paths, "Need a snapshot before the last task completes — run Step 1 and Ctrl-C after task 1 or 2."
ck = str(paths[0])

forked = fork_from(
    ck,
    branch="exec-audience",
    inputs={"audience": "C-suite executives deciding on a pilot budget"},
)
print(forked.tasks[-1].output.raw)
```

Verify new JSON files under `.checkpoints/exec-audience/` (or your branch label). `crewai checkpoint --location ./.checkpoints info <path>` shows lineage.

## 8. Extending — your own provider

Subclass `crewai.state.provider.core.BaseProvider` and implement six methods:

```python
class PostgresProvider(BaseProvider):
    provider_type = "postgres"

    def checkpoint(self, data, location, *, parent_id=None, branch="main") -> str: ...
    async def acheckpoint(self, data, location, *, parent_id=None, branch="main") -> str: ...
    def from_checkpoint(self, location) -> str: ...
    async def afrom_checkpoint(self, location) -> str: ...
    def extract_id(self, location) -> str: ...
    def prune(self, location, max_keep, *, branch="main") -> None: ...
```

Then pass an instance: `CheckpointConfig(provider=PostgresProvider(dsn=...), ...)`. Everything else — the listener, event routing, lineage tracking — works unchanged.

## 9. CLI reference

`--location` is a **group-level** option — it must go *before* the subcommand. The default is `./.checkpoints` (as a JSON dir); if that path doesn't exist but `./.checkpoints.db` does, the CLI auto-picks it.

```
# Bare commands — uses default lookup (./.checkpoints/ or ./.checkpoints.db)
crewai checkpoint list
crewai checkpoint info
crewai checkpoint resume           # most recent
crewai checkpoint resume <ID>

# Custom DB location — pass before the subcommand:
crewai checkpoint --location path/to/my.db list
crewai checkpoint --location path/to/my.db resume <ID>
crewai checkpoint --location path/to/my.db diff <ID1> <ID2>
crewai checkpoint --location path/to/my.db prune
```

`info` shows the serialized `RuntimeState` snapshot — it's the closest thing to a debugger you have when a run behaves unexpectedly.